### Quoted from https://www.kaggle.com/code/ohmammamia/nbayes-logreg-knn-rforest-svm-xgboost  

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#core
import matplotlib.pyplot as plt
%matplotlib inline 
import seaborn as sb
import warnings
import time
warnings.filterwarnings('ignore')
#sklearn
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score, f1_score
from sklearn.metrics import roc_auc_score, plot_confusion_matrix, plot_roc_curve, roc_curve
#models
from sklearn.naive_bayes import GaussianNB          #Naive Bayes
from sklearn.neighbors import KNeighborsClassifier  #KNN
from sklearn.ensemble import RandomForestClassifier #Random Forest
from sklearn.linear_model import LogisticRegression #Logistic Regression
from sklearn.svm import SVC                         #SVM
from sklearn.tree import DecisionTreeClassifier     #Decision Tree
from xgboost import XGBClassifier                   #XGBoost
from lightgbm import LGBMClassifier                 #LightGBM
from catboost import CatBoostClassifier             #CatBoost

In [ ]:
train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv',header=None)
trainLabels = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv',header=None)
test = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv',header=None)

print('Train set shape:', train.shape)
print('Train_labels set shape:', trainLabels.shape)
print('Test set shape:', test.shape)

In [ ]:
print("train\n",train)
print("test\n",test)

In [ ]:
import xgboost as xgb

xgb_params = {
        #'objective': 'multi:softprob',  # 多値分類問題
        #'num_class': 3,                 # 目的変数のクラス数
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'learning_rate': 0.1             # 学習率
    }

In [ ]:
from catboost import CatBoost, CatBoostRegressor, CatBoostClassifier
from catboost import Pool

# Parameters of CatBoostClassifier
params = {
    'loss_function':'CrossEntropy'
}

In [ ]:
kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

max_acc = 0
for i, (idx_train, idx_val) in enumerate(kfold.split(train, trainLabels)):
    X_train, y_train = train.iloc[idx_train], trainLabels.iloc[idx_train]
    X_val, y_val = train.loc[idx_val], trainLabels.iloc[idx_val]
                                                        
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_val, label=y_val)
    
    #xgb = XGBClassifier()
    #xgb_model = xgb.fit(X_train,y_train)
    xgb_model = xgb.train(xgb_params,
                          dtrain,
                          num_boost_round=100,  # 学習ラウンド数は適当
                         )                                                   
    xgb_pred = xgb_model.predict(dtest)
    xgb_pred = np.where(xgb_pred > 0.5, 1, 0)
    
    xgb_accuracy = accuracy_score(y_val, xgb_pred)
    print(xgb_accuracy)
    
    cat = CatBoostClassifier(silent=True)
    model = cat.fit(X_train, y_train)
    y_pred = cat.predict(X_val)
    
    accuracy = accuracy_score(y_val, y_pred)
    
    if accuracy < xgb_accuracy:
        accuracy = xgb_accuracy
        print("Xgb\t")
    print(accuracy)
    
    if max_acc < accuracy:
        max_acc = accuracy
        max_model = model.copy()

In [ ]:
pred = max_model.predict(test)

In [ ]:
pred = pd.DataFrame(pred,columns=['Solution'])

In [ ]:
pred

In [ ]:
pred.index += 1

pred.columns = ['Solution']
pred['Id'] = np.arange(1,pred.shape[0]+1)
pred.to_csv('submission.csv',index=False)